# Stock Market Prediction Using Machine Learning and Ensemble Learning

## Imports

In [96]:
import pandas as pd
import numpy as np
import seaborn as sns
import yfinance as yf
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split,TimeSeriesSplit,cross_validate,RandomizedSearchCV)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,r2_score)


sns.set_theme()

## Data Processing


In [97]:
# Ticker = input(print('Enter Ticker: '))
Ticker ='MSFT'

df = yf.download(Ticker,'2020-01-01')

[*********************100%***********************]  1 of 1 completed


In [98]:
df.head()

Price,Close,High,Low,Open,Volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT
Date,,,,,
2020-01-02,151.829529,151.933509,149.664863,150.090232,22622100
2020-01-03,149.939026,151.196239,149.409676,149.655456,21116200
2020-01-06,150.326584,150.392760,147.944495,148.483307,20813700
2020-01-07,148.955978,150.931594,148.710213,150.600757,21634100
2020-01-08,151.328552,151.999702,149.305671,150.232034,27746500


In [99]:
df['Target'] = df['Close'].shift(-1)
df.head()

Price,Close,High,Low,Open,Volume,Target
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,
Date,,,,,,
2020-01-02,151.829529,151.933509,149.664863,150.090232,22622100,149.939026
2020-01-03,149.939026,151.196239,149.409676,149.655456,21116200,150.326584
2020-01-06,150.326584,150.392760,147.944495,148.483307,20813700,148.955978
2020-01-07,148.955978,150.931594,148.710213,150.600757,21634100,151.328552
2020-01-08,151.328552,151.999702,149.305671,150.232034,27746500,153.219086


In [100]:

df['MA_10'] = df['Close'].rolling(10).mean()
df['MA_50'] = df['Close'].rolling(50).mean()


df['Volatility'] = df['Close'].rolling(10).std()


df['Daily_Return'] = df['Close'].pct_change()

In [101]:
df.dropna(axis=0,inplace=True)

In [102]:
df.describe()

Price,Close,High,Low,Open,Volume,Target,MA_10,MA_50,Volatility,Daily_Return
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,,,,,
count,1611.000000,1611.000000,1611.000000,1611.000000,1.611000e+03,1611.000000,1611.000000,1611.000000,1611.000000,1611.000000
mean,328.667523,331.817069,325.264029,328.598100,2.810566e+07,328.880734,327.692871,324.704804,6.447199,0.001007
std,97.899169,98.437068,97.343279,97.985905,1.292116e+07,97.885035,97.680903,98.041844,4.407729,0.018750
min,128.358337,133.239773,125.609549,129.865402,5.855900e+06,128.358337,135.692841,153.334291,0.841687,-0.147390
25%,244.532082,246.754709,241.438217,243.961222,1.994990e+07,244.636475,244.437048,241.105910,3.866783,-0.008082
50%,319.236450,322.457662,316.378303,319.305429,2.513880e+07,319.642761,318.918939,313.413360,5.551166,0.000846
75%,410.134262,413.558983,406.171621,410.429821,3.266555e+07,410.168869,411.436838,409.659534,7.654965,0.010485
max,538.658508,551.048474,537.366702,550.830186,1.862016e+08,538.658508,522.501855,511.207397,49.547926,0.155067


In [103]:
df.isnull().sum()


Price         Ticker
Close         MSFT      0
High          MSFT      0
Low           MSFT      0
Open          MSFT      0
Volume        MSFT      0
Target                  0
MA_10                   0
MA_50                   0
Volatility              0
Daily_Return            0
dtype: int64

## Feature Selection

In [104]:
target = df['Target']
features = df[['Close','Volume','High','Low','Open',
         'MA_10','MA_50',
         'Volatility']]

## Train Test Split

In [105]:
x_train,x_test,y_train,y_test = train_test_split(features,target,test_size = 0.2,shuffle = False)

## Cross Validation

In [106]:
# TimeSeries Split
tscv = TimeSeriesSplit(n_splits=5)

scoring = {
    'R2':'r2',
    'MAE':'neg_mean_absolute_error',
    'RMSE':'neg_root_mean_squared_error'
}

def run_cross_validation(model,x,y):
    scores = cross_validate(
        model,
        x,
        y,
        cv = tscv,
        scoring = scoring
    )

    print(f"Average R2 : {scores["test_R2"].mean():.2f}")
    print(f"Average MAE : {-scores["test_MAE"].mean():.2f}")
    print(f"Average RMSE : {-scores["test_RMSE"].mean():.2f}")


## Linear Regression

In [107]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

run_cross_validation(lr,x_train,y_train)

Average R2 : 0.96
Average MAE : 4.14
Average RMSE : 5.41


##  Random Forest

In [108]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


run_cross_validation(rf,x_train,y_train)

Average R2 : -0.11
Average MAE : 21.82
Average RMSE : 26.70


## XG Boost Regressor

In [109]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

run_cross_validation(xgb,x_train,y_train)

Average R2 : -0.30
Average MAE : 23.79
Average RMSE : 28.69
